# 04 - Baseline ML DOGE/USDT

## Objetivo del notebook

Construir un baseline de machine learning para predecir la dirección futura del precio de DOGE/USDT utilizando las features generadas en el notebook anterior.

Este notebook servirá como punto de partida para:

- Validar que las features contienen señal útil.

- Establecer una referencia cuantitativa mínima.

- Detectar problemas de leakage o overfitting.

- Comparar modelos más avanzados en fases posteriores.

In [1]:
# ============================================================
# Imports y carga del dataset
# Se carga el dataset enriquecido generado durante el proceso de feature engineering. 
# A partir de este punto, el objetivo ya no es crear variables, sino comprobar si dichas variables contienen suficiente información como para predecir el comportamiento futuro del mercado.
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ============================================================
# Load processed dataset
# ============================================================

DATA_PATH = "../data/processed/DOGEUSDT_5m_binance_2017_2026_features.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["open_time", "close_time"])

print("Dataset loaded")
print("Shape:", df.shape)
print("Date range:", df["open_time"].min(), "->", df["open_time"].max())

display(df.head())

Dataset loaded
Shape: (723348, 44)
Date range: 2019-07-05 13:35:00 -> 2026-05-23 10:10:00


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,...,up_1,future_close_3,future_return_3,up_3,future_close_6,future_return_6,up_6,future_close_12,future_return_12,up_12
0,2019-07-05 13:35:00,0.003815,0.003817,0.003785,0.003809,5726661.0,2019-07-05 13:39:59.999,21762.784020,91,2358281.0,...,1,0.003800,-0.002389,0,0.003822,0.003413,1,0.003863,0.014072,1
1,2019-07-05 13:40:00,0.003812,0.003885,0.003812,0.003840,16701268.0,2019-07-05 13:44:59.999,64066.264668,158,10516524.0,...,0,0.003835,-0.001328,0,0.003836,-0.001198,0,0.003855,0.003776,1
2,2019-07-05 13:45:00,0.003842,0.003880,0.003831,0.003840,21970773.0,2019-07-05 13:49:59.999,84686.627401,137,4464170.0,...,0,0.003831,-0.002344,0,0.003860,0.005287,1,0.003850,0.002787,1
3,2019-07-05 13:50:00,0.003835,0.003860,0.003800,0.003800,6436340.0,2019-07-05 13:54:59.999,24591.400060,90,1357970.0,...,1,0.003822,0.005816,1,0.003919,0.031316,1,0.003867,0.017632,1
4,2019-07-05 13:55:00,0.003820,0.003840,0.003806,0.003835,7182776.0,2019-07-05 13:59:59.999,27471.432703,75,4783851.0,...,0,0.003836,0.000130,1,0.003864,0.007405,1,0.003841,0.001564,1


In [2]:
# ============================================================
# Selección de features y target
# ============================================================

TARGET = "up_1"

# ============================================================
# Explicit columns to exclude
# ============================================================

exclude_cols = [
    "open_time",
    "close_time",

    # Future prices used only to build labels
    "future_close_1",
    "future_close_3",
    "future_close_6",
    "future_close_12",

    # Future returns: targets, never features
    "future_return_1",
    "future_return_3",
    "future_return_6",
    "future_return_12",

    # Binary targets
    "up_1",
    "up_3",
    "up_6",
    "up_12",
]

# ============================================================
# Paranoid exclusion patterns
# ============================================================

exclude_prefixes = (
    "future_",
    "up_",
)

# Old ambiguous target names from previous versions of the notebook.
# If these exist, they are probably future returns and must not be used.
ambiguous_target_cols = [
    "return_1",
    "return_3",
    "return_6",
    "return_12",
]

found_ambiguous_cols = [
    col for col in ambiguous_target_cols
    if col in df.columns
]

if found_ambiguous_cols:
    raise ValueError(
        f"Potential leakage columns found: {found_ambiguous_cols}. "
        "Regenerate the feature engineering dataset using future_return_* names."
    )

# ============================================================
# Feature columns
# ============================================================

feature_cols = [
    col for col in df.columns
    if col not in exclude_cols
    and not col.startswith(exclude_prefixes)
]

X = df[feature_cols]
y = df[TARGET]

print("Target:", TARGET)
print("Number of features:", len(feature_cols))
print()
print(feature_cols)

# Final safety checks
leakage_like_cols = [
    col for col in feature_cols
    if col.startswith("future_")
    or col.startswith("up_")
    or col in ambiguous_target_cols
]

if leakage_like_cols:
    raise ValueError(f"Leakage-like columns detected in features: {leakage_like_cols}")

print("Leakage safety check passed.")

Target: up_1
Number of features: 30

['open', 'high', 'low', 'close', 'volume', 'quote_asset_volume', 'number_of_trades', 'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume', 'return_prev_1', 'log_return_prev_1', 'sma_20', 'ema_10', 'ema_50', 'ema_200', 'ema10_ema50_ratio', 'ema50_ema200_ratio', 'sma20_ema50_ratio', 'volatility_1h', 'zscore_close_1h', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'bb_mid', 'bb_upper', 'bb_lower', 'bb_width', 'bb_percent', 'atr_14']
Leakage safety check passed.


In [3]:
# ============================================================
# Split temporal train/test
# Al ser una serie temporal hacemos un split temporal y no aleatorio, mezclar filas aleatoriamente permitiría entrenar con datos “del futuro” y evaluar con datos “del pasado”, 
# algo completamente irreal en un entorno financiero.El split temporal intenta simular una situación más cercana al mundo real:
# El modelo aprende del pasado, y predice el futuro.
# ============================================================

TEST_SIZE = 0.20

split_index = int(len(df) * (1 - TEST_SIZE))

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print()
print("Train period:")
print(df.iloc[:split_index]["open_time"].min(), "->",
      df.iloc[:split_index]["open_time"].max())

print()
print("Test period:")
print(df.iloc[split_index:]["open_time"].min(), "->",
      df.iloc[split_index:]["open_time"].max())

Train shape: (578678, 30)
Test shape: (144670, 30)

Train period:
2019-07-05 13:35:00 -> 2025-01-06 02:20:00

Test period:
2025-01-06 02:25:00 -> 2026-05-23 10:10:00


In [4]:
# ============================================================
# Naive baseline
# El baseline ingenuo sirve como referencia mínima. Si un modelo de machine learning no mejora claramente esta estrategia, entonces no está aportando valor predictivo suficiente.
# En este caso, el baseline representa una regla muy simple: predecir siempre la clase mayoritaria observada en el conjunto de entrenamiento.
# ============================================================

majority_class = y_train.mode()[0]

naive_predictions = np.full(shape=len(y_test), fill_value=majority_class)

naive_accuracy = accuracy_score(y_test, naive_predictions)
naive_f1 = f1_score(y_test, naive_predictions)

print("Naive baseline")
print("Majority class:", majority_class)
print("Accuracy:", round(naive_accuracy, 4))
print("F1-score:", round(naive_f1, 4))

Naive baseline
Majority class: 0
Accuracy: 0.5106
F1-score: 0.0


El baseline ingenuo obtiene una accuracy aproximada del 51%, lo que indica que las clases están relativamente equilibradas. Sin embargo, el F1-score es 0 porque el modelo predice únicamente la clase mayoritaria (que en nuestro dataset es 0, bajada) y nunca detecta correctamente movimientos de subida.

Esto sirve como referencia importante:

- Accuracy ≈ 51% no implica necesariamente un modelo útil.

- Un modelo puede acertar muchas veces simplemente favoreciendo una clase.

- Por eso métricas como recall y F1-score son especialmente relevantes en clasificación financiera.

In [5]:
# ============================================================
# Logistic Regression Baseline

# Logistic Regression funciona como primer modelo supervisado interpretable. Al ser un modelo lineal, permite comprobar si existe una relación simple entre las features técnicas y la dirección futura del precio.
# En un problema financiero de muy corto plazo, resultados cercanos al azar no son necesariamente un fallo del código, pueden indicar que la señal predictiva es débil frente al ruido del mercado.
# ============================================================

logreg = LogisticRegression(max_iter=1000)

logreg.fit(X_train, y_train)

logreg_preds = logreg.predict(X_test)

# Metrics
logreg_accuracy = accuracy_score(y_test, logreg_preds)
logreg_precision = precision_score(y_test, logreg_preds)
logreg_recall = recall_score(y_test, logreg_preds)
logreg_f1 = f1_score(y_test, logreg_preds)

print("=== Logistic Regression ===")
print("Accuracy :", round(logreg_accuracy, 4))
print("Precision:", round(logreg_precision, 4))
print("Recall   :", round(logreg_recall, 4))
print("F1-score :", round(logreg_f1, 4))

print()
print("Confusion matrix")
print(confusion_matrix(y_test, logreg_preds))

print()
print("Classification report")
print(classification_report(y_test, logreg_preds))

=== Logistic Regression ===
Accuracy : 0.5132
Precision: 0.5104
Recall   : 0.1298
F1-score : 0.207

Confusion matrix
[[65046  8818]
 [61612  9194]]

Classification report
              precision    recall  f1-score   support

           0       0.51      0.88      0.65     73864
           1       0.51      0.13      0.21     70806

    accuracy                           0.51    144670
   macro avg       0.51      0.51      0.43    144670
weighted avg       0.51      0.51      0.43    144670



Logistic Regression mejora ligeramente el baseline ingenuo, alcanzando una accuracy cercana al 51.3% y un F1-score aproximado de 0.21.

La precisión (51%) indica que, cuando el modelo predice una subida, acierta aproximadamente la mitad de las veces. Sin embargo, el recall (13%) muestra que el modelo detecta muy pocas de las subidas reales presentes en el dataset.

Esto sugiere que el modelo es extremadamente conservador: prefiere predecir bajadas y reduce falsos positivos, pero deja escapar gran parte de los movimientos alcistas.

En definitiva, Logistic Regression parece captar una señal muy débil, aunque todavía insuficiente para un sistema de trading robusto.

In [6]:
# ============================================================
# Random Forest Baseline
# Random Forest permite capturar relaciones no lineales entre variables técnicas. Esto puede ser útil cuando la señal no se expresa como una relación lineal simple.
# ============================================================

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)

# Metrics
rf_accuracy = accuracy_score(y_test, rf_preds)
rf_precision = precision_score(y_test, rf_preds)
rf_recall = recall_score(y_test, rf_preds)
rf_f1 = f1_score(y_test, rf_preds)

print("=== Random Forest ===")
print("Accuracy :", round(rf_accuracy, 4))
print("Precision:", round(rf_precision, 4))
print("Recall   :", round(rf_recall, 4))
print("F1-score :", round(rf_f1, 4))

print()
print("Confusion matrix")
print(confusion_matrix(y_test, rf_preds))

print()
print("Classification report")
print(classification_report(y_test, rf_preds))

=== Random Forest ===
Accuracy : 0.5183
Precision: 0.5149
Recall   : 0.2722
F1-score : 0.3561

Confusion matrix
[[55709 18155]
 [51534 19272]]

Classification report
              precision    recall  f1-score   support

           0       0.52      0.75      0.62     73864
           1       0.51      0.27      0.36     70806

    accuracy                           0.52    144670
   macro avg       0.52      0.51      0.49    144670
weighted avg       0.52      0.52      0.49    144670



Random Forest obtiene una accuracy cercana al 52% y un F1-score aproximado de 0.36.

Aunque la mejora parece pequeña en términos absolutos, es relevante comparada con el baseline ingenuo y con Logistic Regression:

- Detecta más patrones no lineales.

- Incrementa notablemente el recall (~27%).

- Mejora el equilibrio entre precisión y capacidad de detección.

El recall representa el porcentaje de movimientos alcistas reales que el modelo consigue identificar correctamente. En este caso, el modelo detecta aproximadamente una cuarta parte de las subidas reales del mercado. El F1-score combina precisión y recall en una sola métrica. Resulta útil cuando interesa equilibrar: no generar demasiadas señales falsas, pero tampoco perder demasiadas oportunidades reales.

El incremento del F1-score respecto a Logistic Regression sugiere que Random Forest está aprovechando mejor la información contenida en las features técnicas.

In [7]:
# ============================================================
# Feature importance
# La importancia de variables ayuda a interpretar qué indicadores influyen más en las predicciones del Random Forest.
# ============================================================

importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf.feature_importances_
})

importance_df = importance_df.sort_values(
    by="importance",
    ascending=False
)

print("Top 20 most important features")

display(importance_df.head(20))

Top 20 most important features


,feature,importance
19,zscore_close_1h,0.101629
28,bb_percent,0.082706
6,number_of_trades,0.072213
10,log_return_prev_1,0.071611
9,return_prev_1,0.061610
5,quote_asset_volume,0.052245
4,volume,0.052149
29,atr_14,0.051833
20,rsi_14,0.035843
8,taker_buy_quote_asset_volume,0.033905


In [8]:
# ============================================================
# XGBoost
# XGBoost suele considerarse uno de los modelos baseline más potentes para datos tabulares y competiciones de machine learning.
# Combina múltiples árboles de decisión de forma secuencial, corrigiendo progresivamente los errores de los árboles anteriores. 
# En datasets financieros suele funcionar especialmente bien cuando existen relaciones complejas y señales débiles mezcladas con mucho ruido.
# ============================================================

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)

xgb_preds = xgb.predict(X_test)

# Metrics
xgb_accuracy = accuracy_score(y_test, xgb_preds)
xgb_precision = precision_score(y_test, xgb_preds, zero_division=0)
xgb_recall = recall_score(y_test, xgb_preds, zero_division=0)
xgb_f1 = f1_score(y_test, xgb_preds, zero_division=0)

print("=== XGBoost ===")
print("Accuracy :", round(xgb_accuracy, 4))
print("Precision:", round(xgb_precision, 4))
print("Recall   :", round(xgb_recall, 4))
print("F1-score :", round(xgb_f1, 4))

print()
print("Confusion matrix")
print(confusion_matrix(y_test, xgb_preds))

print()
print("Classification report")
print(classification_report(y_test, xgb_preds, zero_division=0))

=== XGBoost ===
Accuracy : 0.5169
Precision: 0.5071
Recall   : 0.459
F1-score : 0.4819

Confusion matrix
[[42280 31584]
 [38308 32498]]

Classification report
              precision    recall  f1-score   support

           0       0.52      0.57      0.55     73864
           1       0.51      0.46      0.48     70806

    accuracy                           0.52    144670
   macro avg       0.52      0.52      0.51    144670
weighted avg       0.52      0.52      0.52    144670



XGBoost obtiene una accuracy aproximada del 51.7%, ligeramente inferior a Random Forest, pero mejora de forma clara tanto el recall como el F1-score.

El recall alcanza aproximadamente el 45.9%, lo que significa que el modelo detecta casi la mitad de las subidas reales del conjunto de test. Esta mejora es importante respecto a Logistic Regression y Random Forest, que eran modelos mucho más conservadores a la hora de predecir subidas.

El F1-score, cercano a 0.48, indica un mejor equilibrio entre precisión y recall. Aunque la precisión se mantiene alrededor del 50.7%, el modelo genera una cantidad mayor de señales positivas y consigue capturar más oportunidades reales.

En términos prácticos, XGBoost parece menos conservador que Random Forest: sacrifica una pequeña parte de accuracy global, pero gana capacidad de detección de movimientos alcistas. Para un futuro backtest, este comportamiento puede ser interesante, aunque todavía habría que evaluar si las señales compensan costes, spread y ruido de mercado.

In [9]:
# ============================================================
# Model comparison
# La comparación final permite evaluar si los modelos realmente aportan valor respecto al baseline ingenuo.
# En problemas financieros, pequeñas mejoras sobre el baseline pueden ser relevantes, pero deben interpretarse con cautela.
# Lo importante en esta fase no es demostrar rentabilidad, sino construir una referencia honesta y libre de leakage.
# ============================================================

results = pd.DataFrame([
    {
        "model": "Naive baseline",
        "accuracy": accuracy_score(y_test, naive_predictions),
        "precision": precision_score(y_test, naive_predictions, zero_division=0),
        "recall": recall_score(y_test, naive_predictions, zero_division=0),
        "f1_score": f1_score(y_test, naive_predictions, zero_division=0),
    },
    {
        "model": "Logistic Regression",
        "accuracy": logreg_accuracy,
        "precision": logreg_precision,
        "recall": logreg_recall,
        "f1_score": logreg_f1,
    },
    {
        "model": "Random Forest",
        "accuracy": rf_accuracy,
        "precision": rf_precision,
        "recall": rf_recall,
        "f1_score": rf_f1,
    },
    {
        "model": "XGBoost",
        "accuracy": xgb_accuracy,
        "precision": xgb_precision,
        "recall": xgb_recall,
        "f1_score": xgb_f1,
    }
])

results = results.sort_values(by="f1_score", ascending=False)

print("Model comparison")
display(results)

Model comparison


,model,accuracy,precision,recall,f1_score
3,XGBoost,0.516887,0.507131,0.458972,0.481852
2,Random Forest,0.518290,0.514922,0.272180,0.356121
1,Logistic Regression,0.513168,0.510437,0.129848,0.207030
0,Naive baseline,0.510569,0.000000,0.000000,0.000000


La comparación muestra que todos los modelos supervisados mejoran al baseline ingenuo, aunque la mejora en accuracy es moderada.

El baseline alcanza una accuracy aproximada del 51.1%, pero lo hace prediciendo siempre la clase mayoritaria. Por eso su precision, recall y F1-score para la clase positiva son 0: no detecta ninguna subida.

Logistic Regression mejora ligeramente la accuracy hasta aproximadamente el 51.3%, pero su recall es bajo (13%). Esto indica que detecta pocas subidas reales y se comporta como un modelo muy conservador.

Random Forest obtiene la mayor accuracy global (51.8%) y mejora el F1-score hasta ~0.36, pero mantiene un recall moderado (27.2%). Es decir, acierta algo más en conjunto, pero sigue dejando escapar muchas subidas reales.

XGBoost ofrece el resultado más interesante desde el punto de vista del equilibrio entre métricas: aunque su accuracy baja ligeramente hasta 51.7%, consigue el mejor recall (45.9%) y el mejor F1-score (0.48). Esto significa que detecta muchas más subidas reales que el resto de modelos, aunque a costa de generar más señales positivas.

En conjunto, XGBoost parece el mejor candidato para pasar a una fase posterior de backtesting, porque no solo mejora frente al baseline, sino que ofrece una señal más activa. Aun así, estas métricas siguen siendo modestas y deben interpretarse como evidencia preliminar de señal predictiva, no como prueba de rentabilidad.

In [10]:
# ============================================================
# Save trained models
# Esta celda serializa los modelos entrenados utilizando joblib, permitiendo reutilizarlos posteriormente sin necesidad de volver a entrenar.
# Además del modelo en sí, se guardan también las columnas utilizadas como features, y el target empleado durante el entrenamiento.
# Esto resulta especialmente útil para las siguientes fases del proyecto, como el backtesting, la generación de señales o la integración con un sistema automático de trading.
# ============================================================

import os
import joblib
from pathlib import Path

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

models_to_save = {
    "logistic_regression_doge.joblib": {
        "model": logreg,
        "feature_cols": feature_cols,
        "target": TARGET,
    },

    "random_forest_doge.joblib": {
        "model": rf,
        "feature_cols": feature_cols,
        "target": TARGET,
    },

    "xgboost_doge.joblib": {
        "model": xgb,
        "feature_cols": feature_cols,
        "target": TARGET,
    }
}

for filename, payload in models_to_save.items():
    save_path = MODELS_DIR / filename
    joblib.dump(payload, save_path)
    print(f"Saved: {save_path}")

Saved: ..\models\logistic_regression_doge.joblib
Saved: ..\models\random_forest_doge.joblib
Saved: ..\models\xgboost_doge.joblib
